# Build authority / credibility dataset

Generates the dataset described in `authority/pairs_brainstorm.md`, `authority/design_choice.md`, and `authority/extended.md`.

**Four-condition design** (per `extended.md`) to decompose authority from epistemic accuracy:

|              | plausible claim | dubious claim |
|--------------|-----------------|----------------|
| **authority**     | +A correct      | +A incorrect   |
| **non-authority** | -A correct      | -A incorrect   |

Pair flavors built here:
- **in_domain** — authority vs non-authority in the same domain (`{figure} says {claim}.`)
- **cross_domain** — authority from a different domain vs non-authority (diagnostic: is the direction 'role with credentials' or just 'person with a title'?)
- **anti_authority** — authority vs explicitly unreliable source (max-contrast)
- **institutional** — high-credibility document/org vs low-credibility source (`According to {source}, {claim}.`) — tests whether the direction is general epistemic weight or human-specific
- **multi_source** — single prompt with multiple sources of varying authority (passed through; readout is per-position)

Each pair flavor is generated across both `claim_status=plausible` and `claim_status=dubious`, so the diff-in-means can be decomposed into a pure authority direction and a pure accuracy direction.

Optional final stage uses vLLM to paraphrase the claim sentences for surface diversity (template stays identical so no system-prompt confound is introduced).

## 0. Install + clone repo

Works on Colab / RunPod / any GPU pod. Skip the clone if you're running locally inside the repo.

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ChuloIva/Mech_spoof.git"
REPO_NAME = "Mech_spoof"

def find_repo_root() -> pathlib.Path:
    # 1) running locally inside the repo
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "authority" / "seeds.json").exists():
            return p
    # 2) Colab default
    colab = pathlib.Path("/content") / REPO_NAME
    if (colab / "authority" / "seeds.json").exists():
        return colab
    # 3) clone it
    target = pathlib.Path("/content") / REPO_NAME if pathlib.Path("/content").exists() else pathlib.Path.home() / REPO_NAME
    if not target.exists():
        print(f"Cloning {REPO_URL} -> {target}")
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(target)], check=True)
    return target

REPO_ROOT = find_repo_root()
AUTHORITY_DIR = REPO_ROOT / "authority"
OUTPUT_DIR = AUTHORITY_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT     = {REPO_ROOT}")
print(f"AUTHORITY_DIR = {AUTHORITY_DIR}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")

In [ ]:
# vLLM is only needed for the optional paraphrase stage. Pre-install if running on GPU.
INSTALL_VLLM = False
if INSTALL_VLLM:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=False)

## 1. Config

In [ ]:
# Paraphrase model — any instruction-tuned model works; smaller is fine for sentence rephrasing.
# Defaults are picked to match models already used elsewhere in this repo.
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"   # alternatives: "Qwen/Qwen2.5-7B-Instruct", "meta-llama/Llama-3.3-70B-Instruct"
MAX_MODEL_LEN = 4096
TENSOR_PARALLEL = 1
GPU_MEMORY_UTILIZATION = 0.90

# How many paraphrased variants of each base pair to generate. Set to 0 to skip the vLLM stage.
N_PARAPHRASES = 3

# Sample sizes for derived pair types
N_CROSS_DOMAIN_PER_CLAIM = 2    # cross-domain authorities per (claim, non_authority) — sampled
N_ANTI_AUTHORITY_PER_CLAIM = 2  # anti-authority sources per (claim, authority)
N_INSTITUTIONAL_PER_CLAIM = 2   # (high, low) pairs per claim

RNG_SEED = 0

# Seeds file. "auto" -> use seeds_expanded.json if present, else seeds.json.
# Set to "seeds.json" to force the hand-curated baseline.
SEEDS_FILE = "auto"

# Optional HF token for gated models (Llama)
import getpass
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception:
        os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN") or ""

## 2. Load seeds

In [ ]:
import json, random
from itertools import product

if SEEDS_FILE == "auto":
    expanded_path = AUTHORITY_DIR / "seeds_expanded.json"
    seeds_path = expanded_path if expanded_path.exists() else AUTHORITY_DIR / "seeds.json"
else:
    seeds_path = AUTHORITY_DIR / SEEDS_FILE
print(f"Loading seeds from: {seeds_path.name}")
with open(seeds_path) as f:
    seeds = json.load(f)
with open(AUTHORITY_DIR / "anti_authorities.json") as f:
    anti_data = json.load(f)
with open(AUTHORITY_DIR / "institutional_sources.json") as f:
    inst_data = json.load(f)
with open(AUTHORITY_DIR / "multi_source_prompts.json") as f:
    multi_data = json.load(f)

DOMAINS = list(seeds["domains"].keys())
anti_authorities = anti_data["anti_authorities"]
high_cred = inst_data["high_credibility"]
low_cred = inst_data["low_credibility"]

print(f"Domains: {len(DOMAINS)} -> {DOMAINS}")
for d in DOMAINS:
    blk = seeds["domains"][d]
    print(f"  {d:14s} claims plausible={len(blk['claims_plausible'])} dubious={len(blk['claims_dubious'])} auth={len(blk['authorities'])} non_auth={len(blk['non_authorities'])}")
print(f"\nanti_authorities: {len(anti_authorities)}")
print(f"high_credibility: {len(high_cred)}")
print(f"low_credibility:  {len(low_cred)}")

## 3. Assemble base pairs

Every record is a single source-claim instance. Records with the same `claim_id` differ only in `source` and `source_type`; records with the same `claim_text_id` are paraphrases of the same underlying claim. This lets diff-in-means be computed in any direction:
- group on `claim_id`, contrast `source_type` (authority vs non_authority) → pure authority
- group on `claim_id`, contrast `claim_status` (plausible vs dubious) → pure epistemic accuracy
- group on `pair_type` to slice in_domain / cross_domain / anti_authority / institutional

In [ ]:
rng = random.Random(RNG_SEED)

def _sentence_case(s):
    return s[0].upper() + s[1:] if s else s

def make_record(rec_id, pair_id, claim_id, claim_status, claim, domain, source, source_type, pair_type, template, extras=None):
    raw = template.format(figure=source, claim=claim) if "{figure}" in template else template.format(source=source, claim=claim)
    text = _sentence_case(raw)
    rec = {
        "id": rec_id,
        "pair_id": pair_id,
        "claim_id": claim_id,
        "pair_type": pair_type,
        "domain": domain,
        "claim": claim,
        "claim_status": claim_status,
        "source": source,
        "source_type": source_type,
        "template": template,
        "text": text,
    }
    if extras:
        rec.update(extras)
    return rec

# Build a stable per-claim id so paraphrases / variants line up.
claim_index = {}  # (domain, claim_status, claim_text) -> claim_id
def claim_id_for(domain, claim_status, claim):
    key = (domain, claim_status, claim)
    if key not in claim_index:
        claim_index[key] = len(claim_index)
    return claim_index[key]

records = []
pair_counter = 0
rec_counter = 0

PAIRS_TPL = "{figure} says {claim}."
INST_TPL = "According to {source}, {claim}."

# ----- in_domain ---------------------------------------------------------
for domain in DOMAINS:
    blk = seeds["domains"][domain]
    for claim_status, claim_list in [("plausible", blk["claims_plausible"]), ("dubious", blk["claims_dubious"])]:
        for claim in claim_list:
            cid = claim_id_for(domain, claim_status, claim)
            for auth, non_auth in product(blk["authorities"], blk["non_authorities"]):
                pid = pair_counter; pair_counter += 1
                records.append(make_record(rec_counter, pid, cid, claim_status, claim, domain, auth, "authority", "in_domain", PAIRS_TPL)); rec_counter += 1
                records.append(make_record(rec_counter, pid, cid, claim_status, claim, domain, non_auth, "non_authority", "in_domain", PAIRS_TPL)); rec_counter += 1

n_in_domain = pair_counter
print(f"in_domain pairs:        {n_in_domain}")

In [ ]:
# ----- cross_domain ------------------------------------------------------
# Pair: authority drawn from a DIFFERENT domain than the claim, vs non_authority from the claim's domain.
# If the probe still fires on the cross-domain authority, the direction encodes 'person with a title' not 'person with relevant credentials'.
cross_start = pair_counter
for domain in DOMAINS:
    blk = seeds["domains"][domain]
    other_domains = [d for d in DOMAINS if d != domain]
    for claim_status, claim_list in [("plausible", blk["claims_plausible"]), ("dubious", blk["claims_dubious"])]:
        for claim in claim_list:
            cid = claim_id_for(domain, claim_status, claim)
            # sample mismatched authorities
            mismatched_auths = []
            for other_d in rng.sample(other_domains, k=min(N_CROSS_DOMAIN_PER_CLAIM, len(other_domains))):
                mismatched_auths.append(rng.choice(seeds["domains"][other_d]["authorities"]))
            for x_auth in mismatched_auths:
                non_auth = rng.choice(blk["non_authorities"])
                pid = pair_counter; pair_counter += 1
                extras = {"cross_domain_source_origin": "out_of_domain"}
                records.append(make_record(rec_counter, pid, cid, claim_status, claim, domain, x_auth, "authority", "cross_domain", PAIRS_TPL, extras)); rec_counter += 1
                records.append(make_record(rec_counter, pid, cid, claim_status, claim, domain, non_auth, "non_authority", "cross_domain", PAIRS_TPL)); rec_counter += 1

print(f"cross_domain pairs:     {pair_counter - cross_start}")

In [ ]:
# ----- anti_authority ----------------------------------------------------
# Pair: in-domain authority vs explicitly unreliable source. Same claim. Max-contrast version.
anti_start = pair_counter
for domain in DOMAINS:
    blk = seeds["domains"][domain]
    for claim_status, claim_list in [("plausible", blk["claims_plausible"]), ("dubious", blk["claims_dubious"])]:
        for claim in claim_list:
            cid = claim_id_for(domain, claim_status, claim)
            for _ in range(N_ANTI_AUTHORITY_PER_CLAIM):
                auth = rng.choice(blk["authorities"])
                anti = rng.choice(anti_authorities)
                pid = pair_counter; pair_counter += 1
                records.append(make_record(rec_counter, pid, cid, claim_status, claim, domain, auth, "authority", "anti_authority", PAIRS_TPL)); rec_counter += 1
                records.append(make_record(rec_counter, pid, cid, claim_status, claim, domain, anti, "anti_authority", "anti_authority", PAIRS_TPL)); rec_counter += 1

print(f"anti_authority pairs:   {pair_counter - anti_start}")

In [ ]:
# ----- institutional -----------------------------------------------------
# Non-human sources of varying credibility. Per extended.md, tests whether the direction is general
# 'source credibility / epistemic weight' or human-specific role-authority.
inst_start = pair_counter
for domain in DOMAINS:
    blk = seeds["domains"][domain]
    domain_high = [h for h in high_cred if domain in h["domains"]]
    domain_low  = [l for l in low_cred  if domain in l["domains"]]
    if not domain_high or not domain_low:
        continue
    for claim_status, claim_list in [("plausible", blk["claims_plausible"]), ("dubious", blk["claims_dubious"])]:
        for claim in claim_list:
            cid = claim_id_for(domain, claim_status, claim)
            for _ in range(N_INSTITUTIONAL_PER_CLAIM):
                high = rng.choice(domain_high)["source"]
                low  = rng.choice(domain_low)["source"]
                pid = pair_counter; pair_counter += 1
                records.append(make_record(rec_counter, pid, cid, claim_status, claim, domain, high, "authority", "institutional", INST_TPL)); rec_counter += 1
                records.append(make_record(rec_counter, pid, cid, claim_status, claim, domain, low,  "non_authority", "institutional", INST_TPL)); rec_counter += 1

print(f"institutional pairs:    {pair_counter - inst_start}")
print(f"\nTOTAL records: {len(records)}  /  TOTAL pairs: {pair_counter}")

In [ ]:
# Sanity: count by (pair_type, claim_status, source_type)
from collections import Counter
c = Counter((r["pair_type"], r["claim_status"], r["source_type"]) for r in records)
for k in sorted(c):
    print(f"  {k!s:60s} {c[k]}")

print("\n--- Sample records ---")
for r in rng.sample(records, k=6):
    print(f"[{r['pair_type']:14s} | {r['claim_status']:9s} | {r['source_type']:14s}] {r['text']}")

## 4. Write the base (no-paraphrase) dataset

In [ ]:
BASE_PATH = OUTPUT_DIR / "authority_pairs_base.jsonl"
with open(BASE_PATH, "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")
print(f"Wrote {len(records)} records -> {BASE_PATH}")
print(f"  size: {BASE_PATH.stat().st_size/1024:.1f} KB")

## 5. Multi-source conflict prompts

These are passed through unchanged — they go in as a single user message and the probe is read out at each annotated source span. Per `extended.md`: (Thing 1) does the model's authority gradient match the expected ordering, and (Thing 2) does the response endorse the highest-authority source.

In [ ]:
MULTI_PATH = OUTPUT_DIR / "multi_source_prompts.jsonl"
with open(MULTI_PATH, "w") as f:
    for item in multi_data["prompts"]:
        # add char_start/char_end for each source span for easy probe readout
        annotated = dict(item)
        new_sources = []
        for s in item["sources"]:
            start = item["prompt"].find(s["text"])
            new_sources.append({**s, "char_start": start, "char_end": start + len(s["text"])})
        annotated["sources"] = new_sources
        f.write(json.dumps(annotated) + "\n")
print(f"Wrote {len(multi_data['prompts'])} multi-source prompts -> {MULTI_PATH}")

## 6. (Optional) Paraphrase claim sentences with vLLM

Goal: surface-level diversity in the claim wording without changing what the claim says or who's saying it. The `{figure} says {claim}.` template is preserved — we only rewrite `{claim}`. This keeps the structural design clean while preventing the probe from latching onto exact lexical surface features.

Skip this whole section if `N_PARAPHRASES == 0` or no GPU is available.

In [ ]:
RUN_PARAPHRASE = N_PARAPHRASES > 0
print(f"RUN_PARAPHRASE = {RUN_PARAPHRASE}")

In [ ]:
if RUN_PARAPHRASE:
    from vllm import LLM, SamplingParams
    llm = LLM(
        model=MODEL_ID,
        max_model_len=MAX_MODEL_LEN,
        tensor_parallel_size=TENSOR_PARALLEL,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        trust_remote_code=True,
    )
    tokenizer = llm.get_tokenizer()
    print(f"Loaded {MODEL_ID}")

In [ ]:
PARAPHRASE_SYSTEM = (
    "You rewrite short factual claims into N paraphrases. The rewrites must:\n"
    "- preserve the exact meaning (same factual content, same epistemic status)\n"
    "- be standalone sentence fragments that fit after 'X says ...' grammatically\n"
    "- not add hedging, qualifiers, or extra information\n"
    "- not change who is being referred to (keep 'you', 'your child', etc.)\n"
    "- vary phrasing only at the surface level\n"
    "Return exactly N lines, one paraphrase per line, no numbering, no quoting, no extra commentary."
)

def build_paraphrase_prompt(claim: str, n: int) -> str:
    user = (
        f"Rewrite the following claim into exactly {n} paraphrases. Original claim:\n\n"
        f"\"{claim}\"\n\n"
        f"Output exactly {n} lines."
    )
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": PARAPHRASE_SYSTEM},
         {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

def parse_paraphrases(raw: str, n: int):
    lines = [l.strip(" -•*\"'\t") for l in raw.strip().splitlines() if l.strip()]
    # drop any leading commentary lines that don't look like sentences
    lines = [l for l in lines if len(l) > 4]
    return lines[:n]

In [ ]:
if RUN_PARAPHRASE:
    # one paraphrase generation per unique claim
    unique_claims = sorted({(r["domain"], r["claim_status"], r["claim"], r["claim_id"]) for r in records})
    prompts = [build_paraphrase_prompt(c, N_PARAPHRASES) for (_d, _s, c, _cid) in unique_claims]
    sampling = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=256, n=1)
    print(f"Paraphrasing {len(unique_claims)} unique claims ...")
    outs = llm.generate(prompts, sampling)
    paraphrases_by_claim_id = {}
    for (d, s, c, cid), out in zip(unique_claims, outs):
        paraphrases_by_claim_id[cid] = parse_paraphrases(out.outputs[0].text, N_PARAPHRASES)
    print("Sample paraphrases:")
    for cid in list(paraphrases_by_claim_id)[:3]:
        key = next(k for k in claim_index if claim_index[k] == cid)
        print(f"\n  ORIG ({key[0]}/{key[1]}): {key[2]}")
        for p in paraphrases_by_claim_id[cid]:
            print(f"    -> {p}")

In [ ]:
if RUN_PARAPHRASE:
    expanded = []
    next_id = 0
    for r in records:
        # keep the original
        rec0 = {**r, "id": next_id, "claim_variant": 0, "claim_text_id": f"{r['claim_id']}_v0"}
        expanded.append(rec0); next_id += 1
        # add paraphrases
        for vi, alt_claim in enumerate(paraphrases_by_claim_id.get(r["claim_id"], []), start=1):
            raw = r["template"].format(figure=r["source"], claim=alt_claim) if "{figure}" in r["template"] else r["template"].format(source=r["source"], claim=alt_claim)
            text = _sentence_case(raw)
            recv = {**r, "id": next_id, "claim": alt_claim, "text": text, "claim_variant": vi, "claim_text_id": f"{r['claim_id']}_v{vi}"}
            expanded.append(recv); next_id += 1
    print(f"Expanded records: {len(expanded)} (from {len(records)} base)")

    EXP_PATH = OUTPUT_DIR / "authority_pairs_expanded.jsonl"
    with open(EXP_PATH, "w") as f:
        for r in expanded:
            f.write(json.dumps(r) + "\n")
    print(f"Wrote {EXP_PATH}  ({EXP_PATH.stat().st_size/1024:.1f} KB)")

## 7. Summary

In [ ]:
print("Outputs:")
for p in sorted(OUTPUT_DIR.glob("*.jsonl")):
    n = sum(1 for _ in open(p))
    print(f"  {p.name:42s} {n:6d} lines  {p.stat().st_size/1024:8.1f} KB")

print("\nDownstream use:")
print("  - authority_pairs_base.jsonl      flat records, one per (claim x source) instance.")
print("    Group on pair_id to get the +A/-A pair, or on claim_id to compare across source_type/claim_status.")
print("  - authority_pairs_expanded.jsonl  same schema + claim paraphrases for surface diversity.")
print("  - multi_source_prompts.jsonl      single prompts with annotated source spans for multi-source probe readout.")